# REBA — Entrenamiento de clasificadores en Colab (T4)

**Objetivo:** nuevos miembros para el ensemble cls (fondo/metal/plastico/unicel).

## Pasos
1. Runtime → Change runtime type → **T4 GPU** → guardar.
2. Ejecutar celda 1 (setup), celda 2 (**sube `reba_cls_pack.zip`** cuando lo pida).
3. Ejecutar todo. Al terminar, la última celda descarga `reba_models.zip`.
4. Copiar `reba_models.zip` a la carpeta del proyecto y avisar al agente.

Tiempo estimado en T4: ~60-90 min. Clases (orden alfabético = canónico):
`{0:'fondo', 1:'metal', 2:'plastico', 3:'unicel'}`

In [ ]:
# 1) Setup
!pip -q install ultralytics timm onnx onnxsim onnxruntime-gpu
import torch, platform
print(torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# 2) Subir el pack de datos (elige reba_cls_pack.zip)
from google.colab import files
import os, zipfile, pathlib
os.makedirs('/content/reba', exist_ok=True)
up = files.upload()  # <-- selecciona reba_cls_pack.zip
zname = next(iter(up))
with zipfile.ZipFile(zname) as z:
    z.extractall('/content/reba')
for p in sorted(pathlib.Path('/content/reba').rglob('*')):
    if p.is_dir() and len(p.parts) <= 4:
        print(p)

In [ ]:
# 3) Sanity check del dataset
from pathlib import Path
CLASSES = ['fondo','metal','plastico','unicel']  # orden alfabético = canónico
FULL = Path('/content/reba/cls_esp32_full')
TST  = Path('/content/reba/cls_esp32')
for split in ['train','val']:
    tot = 0
    for c in CLASSES:
        n = len(list((FULL/split/c).glob('*.jpg'))) if (FULL/split/c).is_dir() else 0
        tot += n
        print(f'{split}/{c}: {n}')
    print(f'{split} total: {tot}\n')
for s in ['test_agua','test_tierra','test_agua_limpio','test_tierra_limpio']:
    tot = sum(len(list((TST/s/c).glob('*.jpg'))) for c in CLASSES if (TST/s/c).is_dir())
    print(f'{s}: {tot}')

In [ ]:
# 4) Preproceso EXACTO del server (scale-cover + center crop + /255, sin norm)
import cv2, numpy as np

def cls_preprocess(bgr, th, tw):
    h, w = bgr.shape[:2]
    s = max(th / h, tw / w)
    nw, nh = max(tw, round(w * s)), max(th, round(h * s))
    img = cv2.resize(bgr, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
    x0, y0 = (nw - tw) // 2, (nh - th) // 2
    img = img[y0:y0 + th, x0:x0 + tw, ::-1].astype(np.float32) / 255.0
    return np.ascontiguousarray(img.transpose(2, 0, 1)[None])

def eval_onnx(onnx_path, split, th, tw):
    import onnxruntime as ort
    from collections import Counter
    sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    inp = sess.get_inputs()[0].name
    cm = np.zeros((4,4), int)  # filas=verdad, cols=pred
    per_cls = {c:[0,0] for c in CLASSES}
    for i,c in enumerate(CLASSES):
        d = TST/split/c
        if not d.is_dir(): continue
        for f in d.glob('*.jpg'):
            img = cv2.imread(str(f))
            if img is None: continue
            p = sess.run(None,{inp:cls_preprocess(img,th,tw)})[0][0]
            pred = int(p.argmax())
            cm[i,pred]+=1
            per_cls[c][1]+=1
            per_cls[c][0]+= int(pred==i)
    acc = np.trace(cm)/max(cm.sum(),1)
    per = {c:(round(v[0]/max(v[1],1),3), v[1]) for c,v in per_cls.items()}
    return acc, per, cm

import json, os
os.makedirs('/content/out', exist_ok=True)
RESULTS = {}

In [ ]:
# 5) Modelo A: yolo11m-cls @224 (upgrade directo del cls_11s actual)
from ultralytics import YOLO
m11 = YOLO('yolo11m-cls.pt')
m11.train(data=str(FULL), imgsz=224, epochs=30, batch=128, patience=8,
          optimizer='AdamW', lr0=1e-3, cos_lr=True, label_smoothing=0.1,
          project='/content/runs', name='y11m_cls', pretrained=True, seed=0)
print('val top1:', m11.val().top1)

In [ ]:
# 6) Export yolo11m -> ONNX @224 (incluye softmax + names metadata automático)
best = '/content/runs/y11m_cls/weights/best.pt'
exp = YOLO(best).export(format='onnx', imgsz=224, opset=12, simplify=True)
import shutil, onnx
shutil.copy(exp, '/content/out/cls_11m.onnx')
mm = onnx.load('/content/out/cls_11m.onnx')
print('names meta:', {p.key:p.value for p in mm.metadata_props})

In [ ]:
# 7) Modelo B: yolov8n-cls @160 reentrenado con el dataset FULL (v1 fresco)
v8 = YOLO('yolov8n-cls.pt')
v8.train(data=str(FULL), imgsz=160, epochs=25, batch=192, patience=8,
         optimizer='AdamW', lr0=1e-3, cos_lr=True, label_smoothing=0.1,
         project='/content/runs', name='v8n_cls', pretrained=True, seed=0)
exp = YOLO('/content/runs/v8n_cls/weights/best.pt').export(format='onnx', imgsz=160, opset=12, simplify=True)
shutil.copy(exp, '/content/out/cls_v3.onnx')

In [ ]:
# 8) Modelo C: ConvNeXtV2-Tiny (timm) @224 — diversidad arquitectónica
# Libera la GPU de los modelos YOLO de las celdas 5-7 (OOM fix)
import gc, timm, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path as P
for _v in ['m11','v8']:
    try: del globals()[_v]
    except KeyError: pass
gc.collect(); torch.cuda.empty_cache()
torch.backends.cudnn.benchmark = True

class ClsDS(Dataset):
    def __init__(self, root, th=224, tw=224, train=False, rng=None):
        self.items=[]; self.th=th; self.tw=tw; self.train=train
        self.rng = rng or np.random.default_rng(0)
        for i,c in enumerate(CLASSES):
            for f in P(root).joinpath(c).glob('*.jpg'):
                self.items.append((str(f), i))
    def __len__(self): return len(self.items)
    def __getitem__(self, k):
        f, y = self.items[k]
        img = cv2.imread(f)
        x = cls_preprocess(img, self.th, self.tw)[0]
        if self.train and self.rng.random()<0.5:
            x = x[:, :, ::-1].copy()
        return torch.from_numpy(x), y

tr = ClsDS(FULL/'train', train=True)
va = ClsDS(FULL/'val')
dl_tr = DataLoader(tr, batch_size=64, shuffle=True, num_workers=2, drop_last=True)
dl_va = DataLoader(va, batch_size=128, num_workers=2)
# Normalización ImageNet horneada DENTRO del modelo (input externo = /255,
# igual que cls_preprocess del server — la ONNX exportada es autoconsistente)
class NormNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.core = timm.create_model('convnextv2_tiny', pretrained=True, num_classes=4)
        self.register_buffer('mu', torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer('sd', torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))
    def forward(self, x):
        return self.core((x - self.mu) / self.sd)
net = NormNet().cuda()
opt = torch.optim.AdamW(net.parameters(), lr=2e-4, weight_decay=0.05)
EP=16
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EP*len(dl_tr))
crit = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.cuda.amp.GradScaler()
best_acc, best_state = 0.0, None
for ep in range(EP):
    net.train(); ls=tot=0
    for x,y in dl_tr:
        x,y = x.cuda(), y.cuda()
        with torch.cuda.amp.autocast():
            loss = crit(net(x), y)
        opt.zero_grad(); scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        ls += loss.item()*len(y); tot += len(y)
    net.eval(); ok=n=0
    with torch.no_grad(), torch.cuda.amp.autocast():
        for x,y in dl_va:
            x,y=x.cuda(),y.cuda()
            ok += (net(x).argmax(1)==y).sum().item(); n+=len(y)
    acc=ok/max(n,1)
    if acc>best_acc: best_acc, best_state = acc, {k:v.cpu().clone() for k,v in net.state_dict().items()}
    print(f'ep {ep}: loss={ls/max(tot,1):.4f} val_acc={acc:.4f} best={best_acc:.4f}')
net.load_state_dict(best_state); net.cuda().eval()
print('convnextv2_tiny best val acc:', best_acc)

In [ ]:
# 9) Export ConvNeXt -> ONNX con softmax + names metadata
class Wrap(nn.Module):
    def __init__(self, core): super().__init__(); self.core = core
    def forward(self, x): return F.softmax(self.core(x), dim=1)
wrapped = Wrap(net).eval()
torch.onnx.export(wrapped, torch.zeros(1,3,224,224).cuda(), '/content/out/cls_cnx.onnx',
                  input_names=['images'], output_names=['output0'], opset_version=17,
                  dynamic_axes=None)
!python -m onnxsim /content/out/cls_cnx.onnx /content/out/cls_cnx.onnx
mm = onnx.load('/content/out/cls_cnx.onnx')
pr = mm.metadata_props.add(); pr.key='names'; pr.value=str({0:'fondo',1:'metal',2:'plastico',3:'unicel'})
onnx.save(mm, '/content/out/cls_cnx.onnx')
print('cls_cnx.onnx exportado')

In [ ]:
# 10) Eval de TODOS los ONNX nuevos en los 4 test sets (misma métrica que el server)
for f in sorted(P('/content/out').glob('*.onnx')):
    import onnxruntime as ort
    sess = ort.InferenceSession(str(f), providers=['CPUExecutionProvider'])
    th, tw = (int(v) for v in sess.get_inputs()[0].shape[2:4])
    for s in ['test_agua','test_tierra','test_agua_limpio','test_tierra_limpio']:
        acc, per, cm = eval_onnx(f, s, th, tw)
        RESULTS[f'{f.stem}/{s}'] = {'acc': round(acc,4), 'per_class': per}
        print(f'{f.stem:12s} {s:20s} acc={acc:.3f} {per}')
with open('/content/out/eval_results.json','w') as fh:
    json.dump(RESULTS, fh, indent=1)

In [ ]:
# 11) Cachear probs de los nuevos ONNX sobre los splits de REBA_cls_esp32
# (el meta-head se reentrena en local con estos .npy — formato _feat/{split}_{stem}probs.npy)
import onnxruntime as ort
os.makedirs('/content/out/_feat', exist_ok=True)
PROB_SPLITS = ['train','train_extra','test_agua','test_tierra','test_agua_limpio','test_tierra_limpio']
for f in sorted(P('/content/out').glob('*.onnx')):
    sess = ort.InferenceSession(str(f), providers=['CUDAExecutionProvider','CPUExecutionProvider'])
    inp = sess.get_inputs()[0].name
    th, tw = (int(v) for v in sess.get_inputs()[0].shape[2:4])
    for s in PROB_SPLITS:
        rows = []
        for c in CLASSES:  # orden alfabético = canónico
            d = TST/s/c
            if not d.is_dir(): continue
            for fp in sorted(d.glob('*.jpg')):
                img = cv2.imread(str(fp))
                if img is None: continue
                rows.append(cls_preprocess(img, th, tw)[0])
        if not rows: continue
        X = np.stack(rows)
        probs = np.concatenate([sess.run(None,{inp:X[i:i+256]})[0] for i in range(0,len(X),256)])
        np.save(f'/content/out/_feat/{s}_{f.stem}probs.npy', probs.astype(np.float32))
        print(f'{f.stem} {s}: {probs.shape}')

In [ ]:
# 12) Empaquetar y descargar
import zipfile
with zipfile.ZipFile('/content/reba_models.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in P('/content/out').rglob('*'):
        if f.is_file(): z.write(f, f.relative_to('/content/out'))
print('contenido:', sorted(str(f.relative_to('/content/out')) for f in P('/content/out').rglob('*') if f.is_file()))
files.download('/content/reba_models.zip')

---
**Después:** copia `reba_models.zip` a `D:\Backup_Sebas\UES\Ciclo 6\ARC115\Proyecto\REBA\reba_models.zip` y avísame — yo recalculo las probs cacheadas, reentreno el meta-head del ensemble, evalúo end-to-end y lo dejo desplegado en el server.